### All important libraries

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
import kagglehub


## Download dataset using kaggle api

In [4]:
path = kagglehub.dataset_download("mohitsingh1804/plantvillage")
print("path : ", path)

Using Colab cache for faster access to the 'plantvillage' dataset.
path :  /kaggle/input/plantvillage


### setup saved

In [6]:
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [8]:
Train_path = os.path.join(path,"PlantVillage","train")
Val_path = os.path.join(path,"PlantVillage","val")

## transformation

In [7]:
transform = transforms.Compose(
    [
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
    ]
)

### custom dataset

In [10]:
class MultiClassClassification(Dataset):
  def __init__(self, root_dir, transform=None):
    super().__init__()
    self.samples = []
    self.transform = transform
    self.root_dir = root_dir
    self.classes =sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
    self.class_to_idx = {cls_name : idx for idx, cls_name in enumerate(self.classes)}

    for class_name in self.classes:
      class_path = os.path.join(root_dir, class_name)
      for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)
        if os.path.isfile(img_path):
          label = self.class_to_idx[class_name]
          self.samples.append((img_path, label))

  def __len__(self):
    return len(self.samples)

  def __getitem__(self, index):
    img_path, label = self.samples[index]
    img = Image.open(img_path).convert("RGB")

    if self.transform is not None:
      img = self.transform(img)

    return img, label

### data loader

In [12]:
train_dataset_full = MultiClassClassification(Train_path, transform)
test_dataset_full = MultiClassClassification(Val_path, transform)
num_classes = len(train_dataset_full.classes)

print("Number of classes : ", num_classes)
print("Number of training samples : ", len(train_dataset_full))
print("Number of validation samples : ", len(test_dataset_full))

Number of classes :  38
Number of training samples :  43444
Number of validation samples :  10861


### (optional) use a subset for fast testing

In [63]:
train_dataset = Subset(train_dataset_full, list(range(min(2500,len(train_dataset_full)))))
test_dataset = Subset(test_dataset_full, list(range(min(100,len(test_dataset_full)))))

### use full dataset

In [34]:
# train_dataset = train_dataset_full
# test_dataset = test_dataset_full

In [66]:
pin = True if device.type == 'cuda' else False
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=pin)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=pin)

### Our Custom CNN

In [67]:
class MyCNN(nn.Module):
  def __init__(self, num_classes):
    super().__init__()

    self.features = nn.Sequential(
      nn.Conv2d(3, 32, kernel_size=3, padding='same'),
      nn.ReLU(),
      nn.BatchNorm2d(32),
      nn.MaxPool2d(2),

      nn.Conv2d(32, 64, kernel_size=3, padding='same'),
      nn.ReLU(),
      nn.BatchNorm2d(64),
      nn.MaxPool2d(2),

      nn.Conv2d(64, 128, kernel_size=3, padding='same'),
      nn.ReLU(),
      nn.BatchNorm2d(128),
      nn.MaxPool2d(2)
  )
    self.classifier = nn.Sequential(
      nn.Flatten(),
      nn.Linear(128*16*16, 128),
      nn.ReLU(),
      nn.Dropout(0.4),
      nn.Linear(128, 64),
      nn.ReLU(),
      nn.Dropout(0.4),
      nn.Linear(64, num_classes)
  )

  def forward(self, x):
    x = self.features(x)
    x = self.classifier(x)
    return x

In [68]:
model = MyCNN(num_classes=num_classes).to(device)

### training setup

In [69]:
lerning_rate = 0.001
epochs = 10
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lerning_rate)

### training loop

In [70]:
for epoch in range(epochs):
  model.train()
  total_loss = 0
  for batch_features, batch_labels in train_loader:
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)
    outputs = model(batch_features)
    loss = criterion(outputs, batch_labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss += loss.item()

  avg_loss = total_loss / len(train_loader)
  print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss}")

Epoch 1/10, Loss: 0.8142534766959239
Epoch 2/10, Loss: 0.3953629528419881
Epoch 3/10, Loss: 0.31114068622641927
Epoch 4/10, Loss: 0.2566997726665975
Epoch 5/10, Loss: 0.1522027867716513
Epoch 6/10, Loss: 0.13762049436245202
Epoch 7/10, Loss: 0.16059880275456118
Epoch 8/10, Loss: 0.12100680251723697
Epoch 9/10, Loss: 0.1281696080983192
Epoch 10/10, Loss: 0.1565127701445518


### evaluation

In [71]:
def evaluate(loader):
  model.eval()
  total, correct = 0,0
  with torch.no_grad():
    for batch_features, batch_labels in loader:
      batch_features = batch_features.to(device)
      batch_labels = batch_labels.to(device)
      outputs = model(batch_features)

      _, predicted = torch.max(outputs, 1)
      total += batch_labels.size(0)
      correct += (predicted == batch_labels).sum().item()
  return correct/total

In [72]:
test_acc = evaluate(test_loader)
print(f"Test Accuracy : {test_acc}")

Test Accuracy : 0.91
